# BT para φ²(θ(A,B))

Ejecuta las tres celdas de código en orden. Este cuaderno mide la matriz original y luego la ascendente. Abre los otros cuadernos en pestañas separadas si deseas intentar varias matrices a la vez. Los resultados de Colab deben identificarse como tales.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Selecciona testores_bt_colab.zip
assert 'testores_bt_colab.zip' in uploaded, 'Sube testores_bt_colab.zip'
with zipfile.ZipFile('testores_bt_colab.zip') as z:
    z.extractall('/content')
ROOT = Path('/content/testores_bt_colab')
assert (ROOT / 'CMakeLists.txt').is_file()
print('Proyecto:', ROOT)


In [ ]:
import subprocess
subprocess.run(['cmake', '-S', str(ROOT), '-B', str(ROOT / 'build-colab'), '-DCMAKE_BUILD_TYPE=Release', '-DTESTORES_BUILD_TESTS=OFF'], check=True)
subprocess.run(['cmake', '--build', str(ROOT / 'build-colab'), '--parallel', '2'], check=True)
EXE = ROOT / 'build-colab/testores'
assert EXE.is_file(), 'No se creó el ejecutable de Linux'
print('Ejecutable:', EXE)


## Medir la matriz original y luego la ascendente

Ejecuta esta celda una vez. Muestra el tiempo y descarga un registro tras cada corrida. No hay límite de tiempo dentro del cuaderno; conserva cada registro descargado.


In [ ]:
import csv, re, subprocess, time
from google.colab import files
CASO = 'phi2'
results = ROOT / 'resultados'
results.mkdir(exist_ok=True)
rows = []
for order in ('original', 'ascendente'):
    name = f'{CASO}_theta_AB'
    matrix = ROOT / 'data' / (f'{name}.txt' if order == 'original' else f'ascendente/{name}_ascendente.txt')
    assert matrix.is_file(), matrix
    log = results / f'bt_{CASO}_{order}.txt'
    cmd = [str(EXE), 'bt', '--entrada', str(matrix)]
    print(f'Iniciando {CASO} / {order}', flush=True)
    started = time.monotonic()
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(cmd, cwd=ROOT, stdout=output, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                elapsed = time.monotonic() - started
                print(f'  {elapsed / 60:.1f} min transcurridos', flush=True)
                time.sleep(30)
        except KeyboardInterrupt:
            process.kill()
            process.wait()
            raise
    output_text = log.read_text(encoding='utf-8', errors='replace')
    match = re.search(r'Tiempo total de ejecuci[^\n]*?:\s*([0-9.]+)\s*(µs|ms|s)', output_text)
    if process.returncode == 0 and match:
        value, unit = float(match.group(1)), match.group(2)
        ms = value / 1000 if unit == 'µs' else value * 1000 if unit == 's' else value
        status = 'completo'
        print(f'{CASO} {order}: {value:.3f} {unit} = {ms:.3f} ms', flush=True)
    else:
        ms = ''
        status = f'error_{process.returncode}'
        print(f'{CASO} {order}: sin medición final ({status})', flush=True)
    rows.append((CASO, order, status, ms, log.name))
    summary = results / f'resumen_{CASO}.csv'
    with summary.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(('matriz', 'orden', 'estado', 'tiempo_ms', 'registro'))
        writer.writerows(rows)
    files.download(str(log))
files.download(str(summary))
